# Spark NLP

# Setup

In [1]:
# !pip install spark-nlp==6.4.0
# !pip install pipdeptree

import sparknlp
sparknlp.__version__

'6.4.0'

In [2]:
!pipdeptree -w silence --packages spark-nlp,pyspark

/bin/bash: line 1: pipdeptree: command not found


In [2]:
import socket

hostname = socket.gethostname()
IPAddr = socket.gethostbyname(hostname)

print("Your Computer Name is:", hostname)
print("Your Computer IP Address is:", IPAddr)

Your Computer Name is: 222c441f46bd
Your Computer IP Address is: 172.18.0.4


In [4]:
# # Import Spark NLP
# from sparknlp.base import *
# from sparknlp.annotator import *
# from sparknlp.pretrained import PretrainedPipeline
# import sparknlp

# # Start SparkSession with Spark NLP
# # start() functions has 3 parameters: gpu, apple_silicon, and memory
# # sparknlp.start(gpu=True) will start the session with GPU support
# # sparknlp.start(apple_silicon=True) will start the session with macOS M1 & M2 support
# # sparknlp.start(memory="16G") to change the default driver memory in SparkSession
# spark = sparknlp.start()

# # Download a pre-trained pipeline
# pipeline = PretrainedPipeline('explain_document_dl', lang='en')

# # Your testing dataset
# text = """
# The Mona Lisa is a 16th century oil painting created by Leonardo.
# It's held at the Louvre in Paris.
# """

# # Annotate your testing dataset
# result = pipeline.annotate(text)

# # What's in the pipeline
# print(list(result.keys()))
# # Output: ['entities', 'stem', 'checked', 'lemma', 'document',
# #          'pos', 'token', 'ner', 'embeddings', 'sentence']

# # Check the results
# print(result['entities'])
# # Output: ['Mona Lisa', 'Leonardo', 'Louvre', 'Paris']

# Getting Started - Natural Language Processing with Spark NLP

In [5]:
import sparknlp
import pyspark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as fun
from pyspark.sql.types import *

In [6]:
%matplotlib inline
import matplotlib.pyplot as plt

In [7]:
packages = ','.join([
    "com.johnsnowlabs.nlp:spark-nlp_2.12:6.4.0",
])
jars='spark-nlp-assembly-6.4.0.jar'

spark_conf = SparkConf()
spark_conf = spark_conf.setAppName('spark-nlp-demo')
# spark_conf = spark_conf.setMaster("spark://127.0.0.1:7077").set('spark.driver.host', IPAddr)
spark_conf = spark_conf.setMaster("spark://spark-master:7077").set('spark.driver.host', IPAddr) # same Docker network 
spark_conf = spark_conf.set("spark.jars", "/opt/spark/jars/spark-nlp-assembly-6.4.0.jar")
# spark_conf = spark_conf.set("spark.jars.packages", packages)

# spark_conf = spark_conf.set("spark.jars.packages", packages)
# spark_conf = spark_conf.set("spark.jars", jars)
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
spark.version

# spark.sparkContext.hadoopConfiguration().setClass("fs.file.impl", BareLocalFileSystem.class, FileSystem.class);
# dir(spark.sparkContext)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


'3.5.8'

## Loading and Viewing Data in Apache Spark

In [8]:
# import os
# mini_newsgroups_path = os.path.join('data', 'mini_newsgroups', '*')
# texts = spark.sparkContext.wholeTextFiles(mini_newsgroups_path)
# mock HDFS: root@222c441f46bd:/opt/spark/data# mv /workspaces/learning-cloudnative/codes/compute/spark/data/* .
texts = spark.sparkContext.wholeTextFiles('/opt/spark/data/mini_newsgroups/*')
schema = StructType([
    StructField('filename', StringType()),
    StructField('text', StringType()),
])
texts_df = spark.createDataFrame(texts, schema)
texts_df.show()

+--------------------+--------------------+
|            filename|                text|
+--------------------+--------------------+
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Xref: cantaloupe....|
|file:/opt/spark/d...|Xref: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Xref: cantaloupe....|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Xref: cant

In [9]:
texts_df.limit(5).toPandas()

,filename,text
0,file:/opt/spark/data/mini_newsgroups/alt.athei...,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...
1,file:/opt/spark/data/mini_newsgroups/alt.athei...,Path: cantaloupe.srv.cs.cmu.edu!das-news.harva...
2,file:/opt/spark/data/mini_newsgroups/alt.athei...,Newsgroups: alt.atheism\nPath: cantaloupe.srv....
3,file:/opt/spark/data/mini_newsgroups/alt.athei...,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...
4,file:/opt/spark/data/mini_newsgroups/alt.athei...,Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv....


## Hello World with Spark NLP

In [10]:
texts_df = texts_df.withColumn(
'newsgroup',
fun.split('filename', '/').getItem(7)
)
texts_df.limit(5).toPandas()

,filename,text,newsgroup
0,file:/opt/spark/data/mini_newsgroups/alt.athei...,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,None
1,file:/opt/spark/data/mini_newsgroups/alt.athei...,Path: cantaloupe.srv.cs.cmu.edu!das-news.harva...,None
2,file:/opt/spark/data/mini_newsgroups/alt.athei...,Newsgroups: alt.atheism\nPath: cantaloupe.srv....,None
3,file:/opt/spark/data/mini_newsgroups/alt.athei...,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,None
4,file:/opt/spark/data/mini_newsgroups/alt.athei...,Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv....,None


In [11]:
# online
# from sparknlp.pretrained import PretrainedPipeline
# pipeline = PretrainedPipeline('explain_document_ml', lang='en')
# pipeline.annotate('Hellu wrold!')

# offline
from pyspark.ml import PipelineModel
pipeline = PipelineModel.load("/opt/spark/nlp/models/explain_document_ml_en_4.4.2_3.2_1685186336939/")
from sparknlp.base import LightPipeline
light = LightPipeline(pipeline)
light.annotate("I love deep learning")  

{'document': ['I love deep learning'],
 'spell': ['I', 'love', 'deep', 'learning'],
 'pos': ['PRP', 'VBP', 'JJ', 'VBG'],
 'lemmas': ['I', 'love', 'deep', 'learn'],
 'token': ['I', 'love', 'deep', 'learning'],
 'stems': ['i', 'love', 'deep', 'learn'],
 'sentence': ['I love deep learning']}

In [15]:
spark.stop()

# Cleanup

In [3]:
import socket

hostname = socket.gethostname()
IPAddr = socket.gethostbyname(hostname)

print("Your Computer Name is:", hostname)
print("Your Computer IP Address is:", IPAddr)

import sparknlp
import pyspark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as fun
from pyspark.sql.types import *

spark_conf = SparkConf()
spark_conf = spark_conf.setAppName('spark-nlp-demo')
spark_conf = spark_conf.setMaster("spark://spark-master:7077").set('spark.driver.host', IPAddr) # same Docker network 
spark_conf = spark_conf.set("spark.jars", "/opt/spark/jars/spark-nlp-assembly-6.4.0.jar")
spark_conf = spark_conf.set("spark.driver.extraClassPath", "/opt/spark/jars/spark-nlp-assembly-6.4.0.jar")

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()

texts = spark.sparkContext.wholeTextFiles('/opt/spark/data/mini_newsgroups/alt.atheism/*')
schema = StructType([
    StructField('filename', StringType()),
    StructField('text', StringType()),
])
texts_df = spark.createDataFrame(texts, schema)
texts_df.show()

texts_df = texts_df.withColumn(
'newsgroup',
fun.split('filename', '/').getItem(7)
)
texts_df.limit(5).toPandas()

from pyspark.ml import PipelineModel
pipeline = PipelineModel.load("/opt/spark/nlp/models/explain_document_ml_en_4.4.2_3.2_1685186336939/")
from sparknlp.base import LightPipeline
light = LightPipeline(pipeline)
light.annotate("I love deep learning")  

Your Computer Name is: 222c441f46bd
Your Computer IP Address is: 172.18.0.4


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------------------+--------------------+
|            filename|                text|
+--------------------+--------------------+
|file:/opt/spark/d...|Xref: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Xref: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Xref: cantaloupe....|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Path: cantaloupe....|
|file:/opt/spark/d...|Newsgroups: alt.a...|
|file:/opt/spark/d...|Path: cant

{'document': ['I love deep learning'],
 'spell': ['I', 'love', 'deep', 'learning'],
 'pos': ['PRP', 'VBP', 'JJ', 'VBG'],
 'lemmas': ['I', 'love', 'deep', 'learn'],
 'token': ['I', 'love', 'deep', 'learning'],
 'stems': ['i', 'love', 'deep', 'learn'],
 'sentence': ['I love deep learning']}

In [4]:
spark.stop()